In [ ]:
!pip install "unstructured[pdf]" pymupdf --break-system-packages -q

In [ ]:
import json
import glob
import fitz
from pathlib import Path
from collections import defaultdict
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PDF_PATH    = "/content/drive/MyDrive/punto_eng.pdf"
ICONS_DIR   = "/content/drive/MyDrive/extracted_icons_with_coords/"
OUTPUT_PATH = "/content/drive/MyDrive/chunks.json"

sample running time of following cell: 1 min

In [ ]:
elements = partition_pdf(PDF_PATH, strategy="fast")
print(f"Total elements: {len(elements)}")

pymupdf non è centrato sul pdf, quindi devo prendere il sistema di coordinate (sx in alto)

In [ ]:
def get_page_heights(pdf_path):
    doc = fitz.open(pdf_path)
    heights = {i + 1: page.rect.height for i, page in enumerate(doc)}
    doc.close()
    return heights

page_heights = get_page_heights(PDF_PATH)
print(f"Pages: {len(page_heights)}, sample height: {page_heights[1]:.1f} pts")

In [ ]:
def load_all_icons(icons_dir):
    icons = []
    for path in glob.glob(str(Path(icons_dir) / "*.json")):
        with open(path) as f:
            icons.append(json.load(f))
    return icons

icons = load_all_icons(ICONS_DIR)
print(f"Loaded {len(icons)} icon definitions")

bbox con riferimento da pdf (sx in basso)

In [ ]:
def get_element_bbox(el):
    coords = el.metadata.coordinates
    if coords is None:
        return None
    pts = coords.points
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return (min(xs), min(ys), max(xs), max(ys))

def icon_center_pdfminer(rect, page_height):
    x0, y0, x1, y1 = rect
    cx = (x0 + x1) / 2
    cy = page_height - (y0 + y1) / 2
    return cx, cy

def bbox_contains(bbox, cx, cy):
    x0, y0, x1, y1 = bbox
    return x0 <= cx <= x1 and y0 <= cy <= y1

def center_distance(bbox, cx, cy):
    bx = (bbox[0] + bbox[2]) / 2
    by = (bbox[1] + bbox[3]) / 2
    return ((bx - cx) ** 2 + (by - cy) ** 2) ** 0.5

def edge_distance(bbox, cx, cy):
    x0, y0, x1, y1 = bbox
    dx = max(x0 - cx, 0, cx - x1)
    dy = max(y0 - cy, 0, cy - y1)
    return (dx**2 + dy**2) ** 0.5

In [ ]:
elements_by_page = defaultdict(list)
for el in elements:
    page = el.metadata.page_number
    if page is not None:
        elements_by_page[page].append(el)

print(f"Indexed elements across {len(elements_by_page)} pages")

In [ ]:
MAX_DISTANCE_PT = 120  # skip se è più lontano di così

injected = 0
skipped  = 0

for icon in icons:
    description = icon["description"]

    for occ in icon["occurrences"]:
        page       = occ["page"]
        rect       = occ["rect"]
        page_h     = page_heights.get(page)

        if page_h is None:
            skipped += 1
            continue

        cx, cy = icon_center_pdfminer(rect, page_h)
        candidates = elements_by_page.get(page, [])

        if not candidates:
            skipped += 1
            continue

        best_el   = None
        best_dist = float("inf")

        for el in candidates:
            bbox = get_element_bbox(el)
            if bbox is None:
                continue
            if bbox_contains(bbox, cx, cy):
                best_el   = el
                best_dist = 0
                break
            dist = edge_distance(bbox, cx, cy)
            if dist < best_dist:
                best_dist = dist
                best_el   = el

        if best_el is None or best_dist > MAX_DISTANCE_PT:
            skipped += 1
            continue

        best_el.text = f"[ICON: {description}] " + best_el.text
        injected += 1

print(f"Injected: {injected} | Skipped: {skipped}")

In [ ]:
#check x parametro distanza
none_coords   = 0
too_far       = 0
no_candidates = 0

for icon in icons:
    for occ in icon["occurrences"]:
        page   = occ["page"]
        rect   = occ["rect"]
        page_h = page_heights.get(page)
        if page_h is None:
            continue

        cx, cy     = icon_center_pdfminer(rect, page_h)
        candidates = elements_by_page.get(page, [])

        if not candidates:
            no_candidates += 1
            continue

        bboxes = [get_element_bbox(el) for el in candidates]
        valid  = [(el, b) for el, b in zip(candidates, bboxes) if b is not None]

        if not valid:
            none_coords += 1
            continue

        best_dist = min(edge_distance(b, cx, cy) for _, b in valid)
        if best_dist > MAX_DISTANCE_PT:
            too_far += 1

print(f"No coordinates at all on page:  {none_coords}")
print(f"Coords exist but too far away:  {too_far}")
print(f"No elements on page at all:     {no_candidates}")

In [ ]:
#check x parametro distanza
distances = []

for icon in icons:
    for occ in icon["occurrences"]:
        page   = occ["page"]
        rect   = occ["rect"]
        page_h = page_heights.get(page)
        if page_h is None:
            continue

        cx, cy     = icon_center_pdfminer(rect, page_h)
        candidates = elements_by_page.get(page, [])
        valid      = [(el, get_element_bbox(el)) for el in candidates]
        valid      = [(el, b) for el, b in valid if b is not None]

        if not valid:
            continue

        best_dist = min(edge_distance(b, cx, cy) for _, b in valid)
        distances.append((best_dist, icon["description"], page))

distances.sort()
for dist, name, page in distances:
    print(f"{dist:7.1f} pts  |  page {page:3d}  |  {name}")

In [ ]:
chunks = chunk_by_title(
    elements,
    max_characters=500,
    combine_text_under_n_chars=100,
)

print(f"Total chunks: {len(chunks)}")

In [ ]:
icon_chunks = [c for c in chunks if "[ICON:" in c.text]
print(f"Chunks containing icon descriptions: {len(icon_chunks)}\n")

for c in icon_chunks[:3]:
    print(f"Page {c.metadata.page_number} | {c.text[:300]}")

In [ ]:
def chunk_to_dict(chunk, idx):
    return {
        "chunk_id": f"punto_2015_chunk_{idx:04d}",
        "text": chunk.text,
        "metadata": {
            "page_number": chunk.metadata.page_number,
            "chunk_index": idx,
        }
    }

output = [chunk_to_dict(c, i) for i, c in enumerate(chunks)]

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(output)} chunks → {OUTPUT_PATH}")